In [12]:
"""
AEMO NEM Price Data Downloader
==============================
Downloads 30-minute spot price and demand data for NSW1 region
(which covers the ACT) from AEMO's public aggregated data portal.

Data source:
    https://aemo.com.au/aemo/data/nem/priceanddemand/PRICE_AND_DEMAND_YYYYMM_NSW1.csv

CSV columns:
    REGION       — NEM region (NSW1 for ACT)
    SETTLEMENTDATE — timestamp (end of 30-min interval)
    TOTALDEMAND  — regional demand in MW
    RRP          — Regional Reference Price in $/MWh (the spot price)
    PERIODTYPE   — 'TRADE' for 30-min settlement periods

ACT note:
    The ACT does not have its own NEM region. It falls within the NSW1
    region. The NSW1 spot price applies to all generators and loads
    in both NSW and ACT. For this project, NSW1 prices are the correct
    prices for a community battery in Canberra.

Usage:
    python aemo_downloader.py

    This will download data, save raw CSVs, and produce a cleaned
    parquet/CSV file ready for the DP optimiser.

Requires: pip install pandas requests
"""

import pandas as pd
import requests
import os
import time
from datetime import datetime

In [ ]:
# ============================================================
# Configuration
# ============================================================

REGION = 'NSW1'         # ACT falls within NSW1
DATA_DIR = 'data/aemo'  # directory for downloaded files

# Date range to download
# Start with recent data: Jan 2024 to Dec 2024 (12 months)
START_YEAR = 2024
START_MONTH = 1
END_YEAR = 2024
END_MONTH = 12

# AEMO URL pattern (public, no auth required)
URL_TEMPLATE = (
    'https://aemo.com.au/aemo/data/nem/priceanddemand/'
    'PRICE_AND_DEMAND_{yyyymm}_{region}.csv'
)


In [5]:

# ============================================================
# Download functions
# ============================================================

def download_month(year, month, region=REGION, data_dir=DATA_DIR):
    """
    Download one month of price and demand data from AEMO.

    Args:
        year: e.g. 2024
        month: 1–12
        region: NEM region code (NSW1 for ACT)
        data_dir: directory to save files

    Returns:
        filepath of downloaded CSV, or None if failed
    """
    yyyymm = f"{year}{month:02d}"
    url = URL_TEMPLATE.format(yyyymm=yyyymm, region=region)
    filename = f"PRICE_AND_DEMAND_{yyyymm}_{region}.csv"
    filepath = os.path.join(data_dir, filename)

    # Skip if already downloaded
    if os.path.exists(filepath):
        print(f"  Already exists: {filename}")
        return filepath

    print(f"  Downloading: {url}")
    headers = {
        'User-Agent': 'Mozilla/5.0 (compatible; research-project/1.0)'
    }

    try:
        response = requests.get(url, headers=headers, timeout=30)
        if response.status_code == 200:
            with open(filepath, 'wb') as f:
                f.write(response.content)
            print(f"  Saved: {filename} ({len(response.content) / 1024:.0f} KB)")
            return filepath
        else:
            print(f"  Failed: HTTP {response.status_code}")
            return None
    except Exception as e:
        print(f"  Error: {e}")
        return None


def download_all(start_year=START_YEAR, start_month=START_MONTH,
                 end_year=END_YEAR, end_month=END_MONTH):
    """Download all months in the specified range."""

    os.makedirs(DATA_DIR, exist_ok=True)

    print(f"Downloading AEMO {REGION} price data: "
          f"{start_year}-{start_month:02d} to {end_year}-{end_month:02d}")
    print(f"Saving to: {os.path.abspath(DATA_DIR)}/\n")

    filepaths = []
    year, month = start_year, start_month

    while (year, month) <= (end_year, end_month):
        fp = download_month(year, month)
        if fp:
            filepaths.append(fp)

        # Be polite to AEMO's server
        time.sleep(2)

        # Next month
        month += 1
        if month > 12:
            month = 1
            year += 1

    print(f"\nDownloaded {len(filepaths)} files.")
    return filepaths



In [13]:
# ============================================================
# Data processing
# ============================================================
 
def load_and_clean(filepaths):
    """
    Load downloaded CSVs and produce a clean DataFrame.
 
    AEMO CSV format (aggregated price and demand):
        REGION,SETTLEMENTDATE,TOTALDEMAND,RRP,PERIODTYPE
        NSW1,2024/01/01 00:05:00,6574.92,57.98,TRADE
 
        - Data is at 5-minute intervals
        - RRP is the spot price in $/MWh
        - SETTLEMENTDATE marks the END of each 5-min interval
        - PERIODTYPE is 'TRADE' for all rows
 
    Processing:
        1. Load all monthly CSVs
        2. Resample from 5-minute to 30-minute intervals
           (average 6 prices per half-hour, matching NEM settlement)
        3. Shift timestamp to mark START of interval
 
    Returns:
        DataFrame with columns:
            datetime  — pandas Timestamp (start of 30-min interval)
            price     — average spot price in $/MWh over 30-min period
            demand    — average demand in MW over 30-min period
    """
    dfs = []
    for fp in filepaths:
        try:
            # Read CSV with header row (no skiprows — header is the first row)
            df = pd.read_csv(fp)
 
            # Drop any rows where RRP is not numeric (footer rows, etc.)
            df = df[pd.to_numeric(df['RRP'], errors='coerce').notna()]
 
            dfs.append(df)
        except Exception as e:
            print(f"  Warning: could not read {fp}: {e}")
 
    if not dfs:
        print("No data loaded!")
        return pd.DataFrame()
 
    combined = pd.concat(dfs, ignore_index=True)
 
    # Parse columns
    combined['datetime'] = pd.to_datetime(combined['SETTLEMENTDATE'])
    combined['price'] = combined['RRP'].astype(float)
    combined['demand'] = combined['TOTALDEMAND'].astype(float)
 
    # Sort by time
    combined = combined.sort_values('datetime').reset_index(drop=True)
 
    # AEMO timestamps mark the END of each 5-min interval.
    # Shift back 5 minutes so timestamp = START of interval.
    combined['datetime'] = combined['datetime'] - pd.Timedelta(minutes=5)
 
    # Resample from 5-minute to 30-minute intervals.
    # The NEM settlement price is the average of six 5-minute dispatch prices.
    # Group into 30-min bins and take the mean.
    combined = combined.set_index('datetime')
    resampled = combined[['price', 'demand']].resample('30min').mean()
    resampled = resampled.dropna().reset_index()
 
    print(f"  Raw records: {len(combined)} (5-min intervals)")
    print(f"  Resampled:   {len(resampled)} (30-min intervals)")
 
    return resampled
 

In [7]:
filepaths = download_all()

Saving to: /home/maxwell/Repos/opendss_ex/aemo_data/

  Downloading: https://aemo.com.au/aemo/data/nem/priceanddemand/PRICE_AND_DEMAND_202401_NSW1.csv
  Saved: PRICE_AND_DEMAND_202401_NSW1.csv (398 KB)
  Downloading: https://aemo.com.au/aemo/data/nem/priceanddemand/PRICE_AND_DEMAND_202402_NSW1.csv
  Saved: PRICE_AND_DEMAND_202402_NSW1.csv (373 KB)
  Downloading: https://aemo.com.au/aemo/data/nem/priceanddemand/PRICE_AND_DEMAND_202403_NSW1.csv
  Saved: PRICE_AND_DEMAND_202403_NSW1.csv (398 KB)
  Downloading: https://aemo.com.au/aemo/data/nem/priceanddemand/PRICE_AND_DEMAND_202404_NSW1.csv
  Saved: PRICE_AND_DEMAND_202404_NSW1.csv (387 KB)
  Downloading: https://aemo.com.au/aemo/data/nem/priceanddemand/PRICE_AND_DEMAND_202405_NSW1.csv
  Saved: PRICE_AND_DEMAND_202405_NSW1.csv (403 KB)
  Downloading: https://aemo.com.au/aemo/data/nem/priceanddemand/PRICE_AND_DEMAND_202406_NSW1.csv
  Saved: PRICE_AND_DEMAND_202406_NSW1.csv (390 KB)
  Downloading: https://aemo.com.au/aemo/data/nem/priceandd

In [14]:
# Step 2: Load and clean
print("\nProcessing data...")
df = load_and_clean(filepaths)
if df.empty:
    print("No data processed!")

print(f"  Loaded {len(df)} records")
print(f"  Date range: {df['datetime'].min()} to {df['datetime'].max()}")
print(f"  Price range: ${df['price'].min():.2f} to ${df['price'].max():.2f}/MWh")
print(f"  Mean price: ${df['price'].mean():.2f}/MWh")



Processing data...
  Raw records: 105408 (5-min intervals)
  Resampled:   17568 (30-min intervals)
  Loaded 17568 records
  Date range: 2024-01-01 00:00:00 to 2024-12-31 23:30:00
  Price range: $-515.08 to $17500.00/MWh
  Mean price: $131.02/MWh


In [15]:
def extract_day(df, date_str):
    """
    Extract 48 half-hourly prices for a single day.
 
    Args:
        df: cleaned DataFrame from load_and_clean()
        date_str: date string like '2024-01-15'
 
    Returns:
        numpy array of 48 prices ($/MWh), or None if incomplete
    """
    date = pd.to_datetime(date_str)
    day_data = df[(df['datetime'] >= date) &
                  (df['datetime'] < date + pd.Timedelta(days=1))]
 
    if len(day_data) != 48:
        print(f"  Warning: {date_str} has {len(day_data)} periods (expected 48)")
        if len(day_data) < 40:
            return None
 
    return day_data['price'].values
 
 
def find_interesting_days(df):
    """
    Find days with different price characteristics for analysis.
 
    Returns:
        dict of {label: date_string}
    """
    df = df.copy()
    df['date'] = df['datetime'].dt.date
 
    daily = df.groupby('date').agg(
        mean_price=('price', 'mean'),
        max_price=('price', 'max'),
        min_price=('price', 'min'),
        spread=('price', lambda x: x.max() - x.min()),
        std_price=('price', 'std'),
    )
 
    # Only consider complete days (48 periods)
    counts = df.groupby('date').size()
    complete = counts[counts == 48].index
    daily = daily.loc[daily.index.isin(complete)]
 
    results = {}
 
    # Typical day: closest to median spread
    median_spread = daily['spread'].median()
    typical_idx = (daily['spread'] - median_spread).abs().idxmin()
    results['typical'] = str(typical_idx)
 
    # High spread day: largest price spread (best for arbitrage)
    high_spread_idx = daily['spread'].idxmax()
    results['high_spread'] = str(high_spread_idx)
 
    # Spike day: highest maximum price
    spike_idx = daily['max_price'].idxmax()
    results['spike'] = str(spike_idx)
 
    # Low price day: lowest average price (cheap charging)
    low_idx = daily['mean_price'].idxmin()
    results['low_avg'] = str(low_idx)
 
    # Summer peak: hottest month, highest demand day
    summer_months = df[df['datetime'].dt.month.isin([1, 2, 12])]
    if not summer_months.empty:
        summer_daily = summer_months.groupby(
            summer_months['datetime'].dt.date
        )['demand'].max()
        summer_peak = summer_daily.idxmax()
        results['summer_peak'] = str(summer_peak)
 
    return results


In [ ]:
# ============================================================
# Main
# ============================================================

def main():
    # Step 1: Download data
    filepaths = download_all()

    if not filepaths:
        print("\nNo files downloaded. Check your internet connection.")
        print("You can also download manually from:")
        print("  https://aemo.com.au/aemo/data/nem/priceanddemand/")
        print("  Files named: PRICE_AND_DEMAND_YYYYMM_NSW1.csv")
        return

    # Step 2: Load and clean
    print("\nProcessing data...")
    df = load_and_clean(filepaths)

    if df.empty:
        print("No data processed!")
        return

    print(f"  Loaded {len(df)} records")
    print(f"  Date range: {df['datetime'].min()} to {df['datetime'].max()}")
    print(f"  Price range: ${df['price'].min():.2f} to ${df['price'].max():.2f}/MWh")
    print(f"  Mean price: ${df['price'].mean():.2f}/MWh")

    # Step 3: Save cleaned data
    output_file = os.path.join(DATA_DIR, f'nem_prices_{REGION}_clean.csv')
    df.to_csv(output_file, index=False)
    print(f"\n  Saved cleaned data: {output_file}")

    # Step 4: Find interesting days for analysis
    print("\nInteresting days for DP analysis:")
    days = find_interesting_days(df)
    for label, date in days.items():
        day_prices = extract_day(df, date)
        if day_prices is not None:
            print(f"  {label:<15} {date}  "
                  f"mean=${day_prices.mean():.0f}  "
                  f"min=${day_prices.min():.0f}  "
                  f"max=${day_prices.max():.0f}  "
                  f"spread=${day_prices.max()-day_prices.min():.0f}")

    # Step 5: Export sample days for DP solver
    print("\nExporting sample days for DP solver...")
    for label, date in days.items():
        day_prices = extract_day(df, date)
        if day_prices is not None:
            filename = os.path.join(DATA_DIR, f'prices_{label}_{date}.csv')
            pd.DataFrame({'price_mwh': day_prices}).to_csv(filename, index=False)
            print(f"  {filename}")

    print(f"\nTo use with the DP solver:")
    print(f"  import pandas as pd")
    print(f"  prices = pd.read_csv('aemo_data/prices_typical_YYYY-MM-DD.csv')")
    print(f"  result = solver.solve(prices['price_mwh'].values, initial_soc=100)")

In [17]:
main()

Saving to: /home/maxwell/Repos/opendss_ex/aemo_data/

  Already exists: PRICE_AND_DEMAND_202401_NSW1.csv
  Already exists: PRICE_AND_DEMAND_202402_NSW1.csv
  Already exists: PRICE_AND_DEMAND_202403_NSW1.csv
  Already exists: PRICE_AND_DEMAND_202404_NSW1.csv
  Already exists: PRICE_AND_DEMAND_202405_NSW1.csv
  Already exists: PRICE_AND_DEMAND_202406_NSW1.csv
  Already exists: PRICE_AND_DEMAND_202407_NSW1.csv
  Already exists: PRICE_AND_DEMAND_202408_NSW1.csv
  Already exists: PRICE_AND_DEMAND_202409_NSW1.csv
  Already exists: PRICE_AND_DEMAND_202410_NSW1.csv
  Already exists: PRICE_AND_DEMAND_202411_NSW1.csv
  Already exists: PRICE_AND_DEMAND_202412_NSW1.csv

Downloaded 12 files.

Processing data...
  Raw records: 105408 (5-min intervals)
  Resampled:   17568 (30-min intervals)
  Loaded 17568 records
  Date range: 2024-01-01 00:00:00 to 2024-12-31 23:30:00
  Price range: $-515.08 to $17500.00/MWh
  Mean price: $131.02/MWh

  Saved cleaned data: aemo_data/nem_prices_NSW1_clean.csv

Inter